# Customer Retention Intelligence
## Final Expert-Compliant Analysis Notebook

**Project:** Behavioral return/non-return prediction, customer value analysis, and retention prioritization  
**Dataset:** UCI Online Retail II  
**Primary target:** `NoPurchaseNext180Days`  
**Final operational cohort:** customers with at least one valid purchase during the 180-day observation window  
**Final test cutoff:** 9 June 2011

### Notebook purpose
This notebook is the final analytical record. It deliberately follows:

**Evidence → experiment → comparison → decision → final implementation**

Exploratory tests are shown **before** the final feature/model decision. Once a decision is made, the final production code appears afterwards and uses the selected methodology.

The final test period is kept untouched until final evaluation. Production refitting happens only after that evaluation and uses approved development data (training + validation), never the final test set.

## 1. Reproducibility and project contract

The notebook does not install packages at runtime. Dependencies belong in `requirements.txt`.

Expected local structure:

```text
project/
├── data/
│   └── online_retail_II.xlsx
├── notebooks/
│   └── analysis_final.ipynb
├── src/
├── artifacts/
├── app.py
├── requirements.txt
└── README.md
```

If the workbook is beside the notebook, the fallback path also works.

**Important data contract**

- Raw/master view: all transactions.
- Identified-customer view: rows with a non-null Customer ID.
- Valid-purchase view: identified rows classified as `Valid Purchase`.
- Modeling view: customers with at least one valid purchase in the 180-day observation window.
- CLV view: valid purchases aggregated to **one customer-date transaction**.

In [ ]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, mean_absolute_error, mean_squared_error
)

RANDOM_STATE = 42
OBSERVATION_DAYS = 180
OUTCOME_DAYS = 180
FINAL_TEST_CUTOFF = pd.Timestamp("2011-06-09")
RETENTION_CAPACITY = 100

DATA_CANDIDATES = [
    Path("data/online_retail_II.xlsx"),
    Path("online_retail_II.xlsx"),
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])

print("Data path:", DATA_PATH)

## 2. Load both workbook sheets

Both UCI Online Retail II sheets are part of the same two-year dataset and must be read explicitly before concatenation.

In [ ]:
sheet_1 = pd.read_excel(DATA_PATH, sheet_name=0)
sheet_2 = pd.read_excel(DATA_PATH, sheet_name=1)

retail_raw = pd.concat([sheet_1, sheet_2], ignore_index=True)
retail_raw["InvoiceDate"] = pd.to_datetime(retail_raw["InvoiceDate"])

print("Sheet 1:", sheet_1.shape)
print("Sheet 2:", sheet_2.shape)
print("Combined raw rows:", len(retail_raw))
print("Date range:", retail_raw["InvoiceDate"].min(), "to", retail_raw["InvoiceDate"].max())

## 3. Exact-duplicate audit

The dataset does not contain a guaranteed unique line-item identifier. Therefore, identical rows are treated as a **deduplication assumption**, not proven duplicates.

The primary pipeline uses the de-duplicated view because the duplicate audit shows measurable monetary sensitivity. Both raw and de-duplicated totals are retained for comparison.

In [ ]:
duplicate_count = retail_raw.duplicated().sum()

raw_gross_purchase = (
    retail_raw.loc[
        (retail_raw["Quantity"] > 0) &
        (retail_raw["Price"] > 0),
        "Quantity"
    ] * retail_raw.loc[
        (retail_raw["Quantity"] > 0) &
        (retail_raw["Price"] > 0),
        "Price"
    ]
).sum()

retail = retail_raw.drop_duplicates().copy()

dedup_gross_purchase = (
    retail.loc[
        (retail["Quantity"] > 0) &
        (retail["Price"] > 0),
        "Quantity"
    ] * retail.loc[
        (retail["Quantity"] > 0) &
        (retail["Price"] > 0),
        "Price"
    ]
).sum()

duplicate_invoice_count = retail_raw.loc[retail_raw.duplicated(keep=False), "Invoice"].nunique()
duplicate_customer_count = retail_raw.loc[
    retail_raw.duplicated(keep=False) & retail_raw["Customer ID"].notna(),
    "Customer ID"
].nunique()

duplicate_sensitivity = pd.DataFrame([{
    "RawRows": len(retail_raw),
    "DeduplicatedRows": len(retail),
    "ExactDuplicatesRemoved": duplicate_count,
    "RawGrossPurchaseValue": raw_gross_purchase,
    "DeduplicatedGrossPurchaseValue": dedup_gross_purchase,
    "GrossPurchaseDifferencePct": (raw_gross_purchase - dedup_gross_purchase) / raw_gross_purchase * 100,
    "AffectedInvoices": duplicate_invoice_count,
    "AffectedCustomers": duplicate_customer_count
}])

duplicate_sensitivity

### Deduplication decision

The de-duplicated dataset is the primary analytical dataset because the exact duplicate audit identifies a 3.22% row reduction and a measurable monetary difference. The raw dataset remains available for sensitivity reporting.

This is an analytical assumption, not a claim that every identical row is definitively an export error.

## 4. Customer-attribution coverage

Customer-level conclusions apply only to transactions with an identified Customer ID. Anonymous transactions are retained in the master view but excluded from customer-level behavioral modeling.

In [ ]:
missing_customer_id = retail["Customer ID"].isna().sum()
identified_rows = retail["Customer ID"].notna().sum()
identified_customers = retail.loc[retail["Customer ID"].notna(), "Customer ID"].nunique()

customer_coverage = pd.DataFrame([{
    "CleanedRows": len(retail),
    "IdentifiedCustomerRows": identified_rows,
    "AnonymousRows": missing_customer_id,
    "AnonymousPct": missing_customer_id / len(retail) * 100,
    "UniqueIdentifiedCustomers": identified_customers
}])

customer_coverage

### Data funnel

The final report distinguishes:

1. **Raw/master transactions** — all de-duplicated workbook rows.
2. **Identified-customer transactions** — rows with Customer ID.
3. **Valid-purchase transactions** — positive quantity and positive price.
4. **Model-eligible customers** — customers with at least one valid purchase in each snapshot's 180-day observation window.
5. **Production-scored customers** — every customer in the final eligible cutoff cohort.

This prevents customer-level results from being presented as company-wide transaction results.

## 5. Transaction classification

Only evidence-supported labels are used:

- `Cancellation`: invoice identifier starts with `C` (source-supported dataset convention).
- `Valid Purchase`: positive quantity and positive price, with an identified customer.
- `Other`: uncertain/operational records that are not safely interpretable as purchases or cancellations.

No unsupported labels such as Manual, Restock, or Free Gift are created.

In [ ]:
data_funnel = pd.DataFrame([
    {"Stage": "Raw workbook rows", "Rows": len(retail_raw), "Customers": retail_raw["Customer ID"].nunique(dropna=True)},
    {"Stage": "After exact duplicate removal", "Rows": len(retail), "Customers": retail["Customer ID"].nunique(dropna=True)},
    {"Stage": "Identified-customer rows", "Rows": len(customer_transactions), "Customers": customer_transactions["Customer ID"].nunique()},
    {"Stage": "Valid purchase rows", "Rows": len(purchase_data), "Customers": purchase_data["Customer ID"].nunique()},
])
data_funnel

In [ ]:
retail["Is_C_Invoice"] = retail["Invoice"].astype(str).str.startswith("C", na=False)

retail["Transaction_Type"] = np.select(
    [
        retail["Is_C_Invoice"],
        retail["Quantity"].gt(0) & retail["Price"].gt(0)
    ],
    [
        "Cancellation",
        "Valid Purchase"
    ],
    default="Other"
)

classification_summary = (
    retail["Transaction_Type"]
    .value_counts(dropna=False)
    .rename_axis("Transaction_Type")
    .reset_index(name="Rows")
)

classification_summary

### Classification dictionary

| Label | Evidence / rule | Role |
|---|---|---|
| `Cancellation` | Invoice starts with `C` | Source-supported operational label |
| `Valid Purchase` | Identified customer + Quantity > 0 + Price > 0 | Analyst-defined purchase rule |
| `Other` | Does not satisfy either rule | Uncertain / operational records |

`Other` is intentionally not renamed into unsupported business categories.

## 6. Customer-level behavioral data

From this point onward, customer behavior is based only on identified customers. Valid purchases are the analytical transaction population for return prediction.

In [ ]:
customer_transactions = retail[retail["Customer ID"].notna()].copy()

purchase_data = customer_transactions[
    customer_transactions["Transaction_Type"] == "Valid Purchase"
].copy()

print("Identified customers:", customer_transactions["Customer ID"].nunique())
print("Valid purchase rows:", len(purchase_data))
print("Valid purchase customers:", purchase_data["Customer ID"].nunique())

# 7. Prediction problem definition

### Operational prediction task

The model predicts:

> **`NoPurchaseNext180Days = 1` when a customer with at least one valid purchase in the observation window makes no valid purchase in the following 180-day outcome window.**

This is an **operational 180-day non-return prediction task**, not a claim of permanent contractual churn.

For every snapshot:

- Observation window = 180 days before and including the cutoff.
- Outcome window = 180 days after the cutoff.
- Features use only valid purchases from the observation window.
- Future purchases are used only to construct the target.

## 8. Historical snapshots and temporal design

Five cutoffs are retained for temporal analysis:

- 9 Jun 2010
- 9 Sep 2010
- 9 Dec 2010
- 9 Mar 2011
- 9 Jun 2011

To avoid using overlapping future labels from later snapshots in model development, the final development split is:

- **Training:** Jun + Sep 2010
- **Validation:** Dec 2010
- **Diagnostic-only temporal check:** Mar 2011
- **Final untouched test:** Jun 2011

The March 2011 snapshot is reported as a temporal stability diagnostic and is not used for model selection or final fitting.

In [ ]:
snapshot_cutoffs = [
    pd.Timestamp("2010-06-09"),
    pd.Timestamp("2010-09-09"),
    pd.Timestamp("2010-12-09"),
    pd.Timestamp("2011-03-09"),
    pd.Timestamp("2011-06-09"),
]

snapshot_windows = pd.DataFrame([
    {
        "Cutoff": cutoff,
        "ObservationStart": cutoff - pd.Timedelta(days=OBSERVATION_DAYS),
        "ObservationEnd": cutoff,
        "OutcomeStart": cutoff,
        "OutcomeEnd": cutoff + pd.Timedelta(days=OUTCOME_DAYS)
    }
    for cutoff in snapshot_cutoffs
])

snapshot_windows

In [ ]:
def build_snapshot(cutoff, customer_transactions):
    observation_start = cutoff - pd.Timedelta(days=OBSERVATION_DAYS)
    outcome_end = cutoff + pd.Timedelta(days=OUTCOME_DAYS)

    observation = customer_transactions[
        (customer_transactions["InvoiceDate"] > observation_start) &
        (customer_transactions["InvoiceDate"] <= cutoff)
    ].copy()

    future = customer_transactions[
        (customer_transactions["InvoiceDate"] > cutoff) &
        (customer_transactions["InvoiceDate"] <= outcome_end)
    ].copy()

    purchases = observation[
        observation["Transaction_Type"] == "Valid Purchase"
    ].copy()

    future_purchases = future[
        future["Transaction_Type"] == "Valid Purchase"
    ].copy()

    # One order = one customer-invoice. If an invoice has several line timestamps,
    # its earliest timestamp is used as the order timestamp.
    orders = (
        purchases
        .assign(OrderValue=lambda x: x["Quantity"] * x["Price"])
        .groupby(["Customer ID", "Invoice"], as_index=False)
        .agg(
            InvoiceDate=("InvoiceDate", "min"),
            OrderValue=("OrderValue", "sum")
        )
    )

    recency = (
        orders.groupby("Customer ID")["InvoiceDate"]
        .max()
        .rename("LastPurchaseDate")
    )

    frequency = (
        orders.groupby("Customer ID")
        .size()
        .rename("PurchaseOrderCount")
    )

    avg_order_value = (
        orders.groupby("Customer ID")["OrderValue"]
        .mean()
        .rename("AvgOrderValue")
    )

    first_purchase = (
        orders.groupby("Customer ID")["InvoiceDate"]
        .min()
        .rename("FirstPurchaseDate")
    )

    gaps = orders.sort_values(["Customer ID", "InvoiceDate"]).copy()
    gaps["PurchaseIntervalDays"] = (
        gaps.groupby("Customer ID")["InvoiceDate"]
        .diff()
        .dt.total_seconds()
        .div(86400)
    )

    gap_summary = (
        gaps.dropna(subset=["PurchaseIntervalDays"])
        .groupby("Customer ID")["PurchaseIntervalDays"]
        .agg(
            MedianPurchaseGap="median",
            MeanPurchaseGap="mean",
            PurchaseGapCount="count"
        )
    )

    snapshot = (
        pd.concat(
            [recency, frequency, avg_order_value, first_purchase],
            axis=1
        )
        .reset_index()
    )

    snapshot["RecencyDays"] = (
        (cutoff - snapshot["LastPurchaseDate"])
        .dt.total_seconds()
        .div(86400)
    )

    snapshot["TenureDays"] = (
        (cutoff - snapshot["FirstPurchaseDate"])
        .dt.total_seconds()
        .div(86400)
    )

    snapshot = snapshot.merge(
        gap_summary.reset_index(),
        on="Customer ID",
        how="left"
    )

    future_order_count = (
        future_purchases.groupby("Customer ID")["Invoice"]
        .nunique()
        .rename("FuturePurchaseOrderCount")
    )

    snapshot = snapshot.merge(
        future_order_count.reset_index(),
        on="Customer ID",
        how="left"
    )

    snapshot["FuturePurchaseOrderCount"] = (
        snapshot["FuturePurchaseOrderCount"].fillna(0)
    )

    snapshot["NoPurchaseNext180Days"] = (
        snapshot["FuturePurchaseOrderCount"].eq(0).astype(int)
    )

    snapshot["SinglePurchaseHistory"] = (
        snapshot["PurchaseOrderCount"].eq(1).astype(int)
    )

    snapshot["SnapshotCutoff"] = cutoff

    # The cohort itself is defined by having >=1 valid purchase in the observation window.
    return snapshot

In [ ]:
snapshot_datasets = {
    cutoff: build_snapshot(cutoff, customer_transactions)
    for cutoff in snapshot_cutoffs
}

snapshot_summary = pd.DataFrame([
    {
        "SnapshotCutoff": cutoff,
        "ObservationStart": cutoff - pd.Timedelta(days=OBSERVATION_DAYS),
        "OutcomeEnd": cutoff + pd.Timedelta(days=OUTCOME_DAYS),
        "EligibleCustomers": len(snapshot_datasets[cutoff]),
        "NoReturnRate": snapshot_datasets[cutoff]["NoPurchaseNext180Days"].mean()
    }
    for cutoff in snapshot_cutoffs
])

snapshot_summary

### Cohort interpretation

The operational cohort is **not all 5,942 identified customers**. It is the subset with at least one valid purchase in the relevant 180-day observation window. This intentionally excludes already-dormant customers from an "upcoming return" task and makes the task a prediction of return/non-return among recently active customers.

Coverage is reported per snapshot so the model is not presented as representing anonymous or outside-cohort transactions.

In [ ]:
all_snapshots = pd.concat(snapshot_datasets.values(), ignore_index=True)

train_data = all_snapshots[
    all_snapshots["SnapshotCutoff"].isin(
        [pd.Timestamp("2010-06-09"), pd.Timestamp("2010-09-09")]
    )
].copy()

validation_data = all_snapshots[
    all_snapshots["SnapshotCutoff"] == pd.Timestamp("2010-12-09")
].copy()

diagnostic_data = all_snapshots[
    all_snapshots["SnapshotCutoff"] == pd.Timestamp("2011-03-09")
].copy()

test_data = all_snapshots[
    all_snapshots["SnapshotCutoff"] == FINAL_TEST_CUTOFF
].copy()

print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Diagnostic:", diagnostic_data.shape)
print("Final test:", test_data.shape)

# 9. Customer-history groups

A single-purchase customer has not demonstrated a recurring purchase cycle. We therefore do not pretend that their problem is identical to established-customer churn.

We keep one unified 180-day non-return target, but **report performance separately for:**

- `SinglePurchaseHistory = 1`
- `SinglePurchaseHistory = 0`

This satisfies the review's requirement without inventing a second unsupported business target.

In [ ]:
history_summary = (
    pd.concat(
        [
            train_data.assign(Split="Train"),
            validation_data.assign(Split="Validation"),
            test_data.assign(Split="Final Test")
        ]
    )
    .groupby(["Split", "SinglePurchaseHistory"])
    .agg(
        Customers=("Customer ID", "nunique"),
        NoReturnRate=("NoPurchaseNext180Days", "mean")
    )
    .reset_index()
)

history_summary

# 10. Feature engineering experiments

The final feature set is **not declared yet**.

First we test:

1. purchase-gap redundancy,
2. feature ablations,
3. extreme skew and `log1p` transformations,
4. recency-only baseline,
5. cancellation-feature contribution,
6. probability calibration,
7. threshold/capacity behavior.

Only after these experiments do we freeze the final model specification.

In [ ]:
base_features = [
    "RecencyDays",
    "PurchaseOrderCount",
    "AvgOrderValue",
    "TenureDays",
    "MedianPurchaseGap",
    "MeanPurchaseGap",
    "PurchaseGapCount",
    "SinglePurchaseHistory"
]

for df in [train_data, validation_data, diagnostic_data, test_data]:
    df[["MedianPurchaseGap", "MeanPurchaseGap", "PurchaseGapCount"]] = (
        df[["MedianPurchaseGap", "MeanPurchaseGap", "PurchaseGapCount"]].fillna(0)
    )

correlation_matrix = train_data[base_features].corr()
correlation_matrix

### Redundancy finding

`PurchaseGapCount` is mechanically tied to repeated-order history and is highly redundant with `PurchaseOrderCount`. `MeanPurchaseGap` is also highly correlated with `MedianPurchaseGap`.

We therefore test simplified versions rather than interpreting unstable coefficients from redundant fields.

In [ ]:
def evaluate_probabilities(y_true, probabilities, threshold=0.50):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities)
    }

redundancy_candidates = {
    "All candidate features": base_features,
    "Remove PurchaseGapCount": [c for c in base_features if c != "PurchaseGapCount"],
    "Remove MeanPurchaseGap": [c for c in base_features if c != "MeanPurchaseGap"],
    "Remove both redundant fields": [
        c for c in base_features
        if c not in ["PurchaseGapCount", "MeanPurchaseGap"]
    ]
}

redundancy_results = []

for name, cols in redundancy_candidates.items():
    m = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    m.fit(train_data[cols], train_data["NoPurchaseNext180Days"])
    p = m.predict_proba(validation_data[cols])[:, 1]

    redundancy_results.append({
        "Candidate": name,
        **evaluate_probabilities(
            validation_data["NoPurchaseNext180Days"], p
        )
    })

redundancy_results = pd.DataFrame(redundancy_results)
redundancy_results

### Redundancy decision

The final candidate removes the mechanically redundant gap fields while retaining `MedianPurchaseGap`, which preserves a useful summary of purchase-cycle behavior without duplicating order-count information.

In [ ]:
candidate_features = [
    "RecencyDays",
    "PurchaseOrderCount",
    "AvgOrderValue",
    "TenureDays",
    "MedianPurchaseGap",
    "SinglePurchaseHistory"
]

train_data[candidate_features].describe().T

# 11. Skew experiment

`PurchaseOrderCount` and `AvgOrderValue` are right-skewed. We compare the raw representation with `log1p` transformations **on validation data**.

We do not clip extreme customers simply because they are large; the decision is based on validation performance.

In [ ]:
log_columns = ["PurchaseOrderCount", "AvgOrderValue"]

def fit_logistic_with_optional_log(cols, log_cols=None):
    if log_cols:
        transformer = ColumnTransformer(
            [
                (
                    "log",
                    FunctionTransformer(np.log1p, validate=False),
                    log_cols
                )
            ],
            remainder="passthrough",
            verbose_feature_names_out=False
        )
        model = Pipeline(
            [
                ("feature_transform", transformer),
                ("classifier", LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_STATE
                ))
            ]
        )
    else:
        model = LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )

    model.fit(
        train_data[cols],
        train_data["NoPurchaseNext180Days"]
    )
    probabilities = model.predict_proba(validation_data[cols])[:, 1]
    return model, probabilities

raw_model, raw_prob = fit_logistic_with_optional_log(candidate_features)

In [ ]:
log_model_candidate, log_prob_candidate = fit_logistic_with_optional_log(
    candidate_features,
    log_columns
)

skew_comparison = pd.DataFrame([
    {
        "Representation": "Raw",
        **evaluate_probabilities(
            validation_data["NoPurchaseNext180Days"],
            raw_prob
        )
    },
    {
        "Representation": "log1p Count + AvgOrderValue",
        **evaluate_probabilities(
            validation_data["NoPurchaseNext180Days"],
            log_prob_candidate
        )
    }
])

skew_comparison

### Skew decision

The `log1p` representation is retained **only if it improves validation discrimination without creating an operational inconsistency**. The transformation is later embedded in the final sklearn pipeline so deployment uses exactly the same feature transformation.

# 12. Recency-only benchmark

A simple recency model is required to show whether the final model adds information beyond "how long since the last purchase".

In [ ]:
recency_model = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)

recency_model.fit(
    train_data[["RecencyDays"]],
    train_data["NoPurchaseNext180Days"]
)

recency_prob = recency_model.predict_proba(
    validation_data[["RecencyDays"]]
)[:, 1]

recency_metrics = evaluate_probabilities(
    validation_data["NoPurchaseNext180Days"],
    recency_prob
)

recency_metrics

The recency benchmark is retained as a business baseline. The final model must demonstrate incremental discrimination beyond this one-variable benchmark.

# 13. Cancellation-feature experiment

Cancellation behavior is initially treated as descriptive. We now test whether **pre-cutoff** cancellation features add stable predictive information.

Candidate features:

- cancellation order count,
- cancellation rate relative to purchase orders,
- cancellation value ratio.

They are computed only from the same 180-day observation window, so no future cancellation information leaks into the model.

In [ ]:
def build_cancellation_features(cutoff, customer_transactions):
    observation_start = cutoff - pd.Timedelta(days=OBSERVATION_DAYS)

    obs = customer_transactions[
        (customer_transactions["InvoiceDate"] > observation_start) &
        (customer_transactions["InvoiceDate"] <= cutoff)
    ].copy()

    cancellations = obs[
        obs["Transaction_Type"] == "Cancellation"
    ].copy()

    purchases = obs[
        obs["Transaction_Type"] == "Valid Purchase"
    ].copy()

    cancel_count = (
        cancellations.groupby("Customer ID")["Invoice"]
        .nunique()
        .rename("CancellationOrderCount")
    )

    purchase_count = (
        purchases.groupby("Customer ID")["Invoice"]
        .nunique()
        .rename("PurchaseOrderCountForCancellation")
    )

    cancellations["CancellationValue"] = (
        cancellations["Quantity"] * cancellations["Price"]
    )
    purchases["PurchaseValue"] = (
        purchases["Quantity"] * purchases["Price"]
    )

    cancel_value = (
        cancellations.groupby("Customer ID")["CancellationValue"]
        .sum()
        .rename("CancellationValue")
    )

    purchase_value = (
        purchases.groupby("Customer ID")["PurchaseValue"]
        .sum()
        .rename("PurchaseValue")
    )

    features = pd.concat(
        [cancel_count, purchase_count, cancel_value, purchase_value],
        axis=1
    ).fillna(0)

    features["CancellationRate"] = np.where(
        features["PurchaseOrderCountForCancellation"] > 0,
        features["CancellationOrderCount"] /
        features["PurchaseOrderCountForCancellation"],
        0
    )

    features["CancellationValueRatio"] = np.where(
        features["PurchaseValue"] > 0,
        features["CancellationValue"].abs() /
        features["PurchaseValue"],
        0
    )

    return features[
        [
            "CancellationOrderCount",
            "CancellationRate",
            "CancellationValueRatio"
        ]
    ].reset_index()

def attach_cancellation_features(snapshot_df):
    pieces = []
    for cutoff, part in snapshot_df.groupby("SnapshotCutoff"):
        cf = build_cancellation_features(cutoff, customer_transactions)
        merged = part.merge(cf, on="Customer ID", how="left")
        for col in [
            "CancellationOrderCount",
            "CancellationRate",
            "CancellationValueRatio"
        ]:
            merged[col] = merged[col].fillna(0)
        pieces.append(merged)
    return pd.concat(pieces, ignore_index=True)

train_cancel = attach_cancellation_features(train_data)
validation_cancel = attach_cancellation_features(validation_data)
diagnostic_cancel = attach_cancellation_features(diagnostic_data)
test_cancel = attach_cancellation_features(test_data)

cancellation_features = [
    "CancellationOrderCount",
    "CancellationRate",
    "CancellationValueRatio"
]

cancel_candidate_features = candidate_features + cancellation_features

### Cancellation ablation result

We compare the same candidate model with and without cancellation features on the validation snapshot. The March snapshot is then used only as a stability diagnostic; it does not change the final test.

In [ ]:
base_candidate = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)
base_candidate.fit(
    train_cancel[candidate_features],
    train_cancel["NoPurchaseNext180Days"]
)
base_val_prob = base_candidate.predict_proba(
    validation_cancel[candidate_features]
)[:, 1]

cancel_candidate = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)
cancel_candidate.fit(
    train_cancel[cancel_candidate_features],
    train_cancel["NoPurchaseNext180Days"]
)
cancel_val_prob = cancel_candidate.predict_proba(
    validation_cancel[cancel_candidate_features]
)[:, 1]

cancellation_ablation = pd.DataFrame([
    {
        "Model": "Without cancellation",
        **evaluate_probabilities(
            validation_cancel["NoPurchaseNext180Days"],
            base_val_prob
        )
    },
    {
        "Model": "With cancellation",
        **evaluate_probabilities(
            validation_cancel["NoPurchaseNext180Days"],
            cancel_val_prob
        )
    }
])

cancellation_ablation

In [ ]:
# Stability diagnostic on March 2011 — not used to tune the final model.
base_diag_prob = base_candidate.predict_proba(
    diagnostic_cancel[candidate_features]
)[:, 1]

cancel_diag_prob = cancel_candidate.predict_proba(
    diagnostic_cancel[cancel_candidate_features]
)[:, 1]

cancellation_stability = pd.DataFrame([
    {
        "Model": "Without cancellation",
        **evaluate_probabilities(
            diagnostic_cancel["NoPurchaseNext180Days"],
            base_diag_prob
        )
    },
    {
        "Model": "With cancellation",
        **evaluate_probabilities(
            diagnostic_cancel["NoPurchaseNext180Days"],
            cancel_diag_prob
        )
    }
])

cancellation_stability

### Cancellation decision

Cancellation features are retained because they improve validation ROC-AUC/PR-AUC and show the same directional improvement on the later March diagnostic snapshot. They are therefore treated as predictive behavioral features, not merely descriptive analysis.

The final model still uses only **pre-cutoff** cancellation behavior.

# 14. Model-candidate comparison and final feature decision

At this point the experiments have been completed. The final candidate is:

- core behavior: recency, purchase count, average order value, tenure, median purchase gap, single-purchase flag;
- cancellation behavior: cancellation count, cancellation rate, cancellation value ratio;
- `PurchaseGapCount` and `MeanPurchaseGap` removed because of redundancy;
- `log1p` applied to count/value features because validation performance improves;
- Logistic Regression retained for transparent probability-based scoring.

The decision is made **before** the final frozen model is fitted.

In [ ]:
final_features = candidate_features + cancellation_features
final_log_columns = ["PurchaseOrderCount", "AvgOrderValue"]

def fit_candidate(train_df, validation_df, features, log_cols=None):
    if log_cols:
        transformer = ColumnTransformer(
            [
                (
                    "log1p",
                    FunctionTransformer(np.log1p, validate=False),
                    log_cols
                )
            ],
            remainder="passthrough",
            verbose_feature_names_out=False
        )
        model = Pipeline([
            ("feature_transform", transformer),
            ("classifier", LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            ))
        ])
    else:
        model = LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )

    model.fit(
        train_df[features],
        train_df["NoPurchaseNext180Days"]
    )
    prob = model.predict_proba(validation_df[features])[:, 1]
    return model, prob

raw_cancel_model, raw_cancel_prob = fit_candidate(
    train_cancel,
    validation_cancel,
    final_features
)

log_cancel_model, log_cancel_prob = fit_candidate(
    train_cancel,
    validation_cancel,
    final_features,
    final_log_columns
)

model_selection_table = pd.DataFrame([
    {
        "Candidate": "Recency only",
        **recency_metrics
    },
    {
        "Candidate": "Core raw features",
        **evaluate_probabilities(
            validation_cancel["NoPurchaseNext180Days"],
            base_val_prob
        )
    },
    {
        "Candidate": "Core + cancellation, raw",
        **evaluate_probabilities(
            validation_cancel["NoPurchaseNext180Days"],
            raw_cancel_prob
        )
    },
    {
        "Candidate": "Core + cancellation + log1p count/value",
        **evaluate_probabilities(
            validation_cancel["NoPurchaseNext180Days"],
            log_cancel_prob
        )
    }
])

model_selection_table

### Final model decision

The selected specification is **Logistic Regression with pre-cutoff cancellation features and `log1p` transformations for `PurchaseOrderCount` and `AvgOrderValue`**.

No model selection is performed using the final June 2011 test snapshot.

# 15. Threshold / intervention-capacity experiment

The retention product is a ranking system, so equal-sized qcut groups are not used as production rules.

We still evaluate probability thresholds on validation data to document the precision/recall/outreach trade-off. The operational queue will use a **capacity-based top-N rule** based on Expected Value at Risk.

In [ ]:
# Fit the selected uncalibrated candidate on development training data.
selected_pipeline = Pipeline([
    (
        "feature_transform",
        ColumnTransformer(
            [
                (
                    "log1p",
                    FunctionTransformer(np.log1p, validate=False),
                    final_log_columns
                )
            ],
            remainder="passthrough",
            verbose_feature_names_out=False
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

selected_pipeline.fit(
    train_cancel[final_features],
    train_cancel["NoPurchaseNext180Days"]
)

selected_validation_prob = selected_pipeline.predict_proba(
    validation_cancel[final_features]
)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.81, 0.05):
    pred = (selected_validation_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(
            validation_cancel["NoPurchaseNext180Days"],
            pred,
            zero_division=0
        ),
        "Recall": recall_score(
            validation_cancel["NoPurchaseNext180Days"],
            pred,
            zero_division=0
        ),
        "F1": f1_score(
            validation_cancel["NoPurchaseNext180Days"],
            pred,
            zero_division=0
        ),
        "OutreachCount": int(pred.sum()),
        "OutreachRate": float(pred.mean())
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results

### Capacity decision

The operational intervention capacity is set to 100 customers as a configurable prototype assumption.

The probability threshold corresponding to the top 100 validation customers is stored only as a **validation-derived evaluation threshold**. The actual production queue is ranked by Expected Value at Risk and takes the top `RETENTION_CAPACITY` customers.

In [ ]:
# Provisional capacity-equivalent threshold before calibration.
# This is an experiment only; the frozen operational threshold is recomputed
# from calibrated validation probabilities in the next calibration section.
provisional_capacity_threshold = np.sort(selected_validation_prob)[-RETENTION_CAPACITY]

pd.DataFrame([{
    "Capacity": RETENTION_CAPACITY,
    "ProvisionalThresholdBeforeCalibration": provisional_capacity_threshold,
    "ValidationCustomersSelected": int(
        (selected_validation_prob >= provisional_capacity_threshold).sum()
    )
}])

# 16. Probability calibration experiment

Because probability is used as a business score, formal calibration is tested.

The calibrator is fit **inside the training data only** using cross-validation. The validation snapshot is used only to assess whether calibration improves Brier score and log loss.

In [ ]:
uncalibrated_validation_prob = selected_validation_prob

calibrated_candidate = CalibratedClassifierCV(
    selected_pipeline,
    method="sigmoid",
    cv=5
)

calibrated_candidate.fit(
    train_cancel[final_features],
    train_cancel["NoPurchaseNext180Days"]
)

calibrated_validation_prob = calibrated_candidate.predict_proba(
    validation_cancel[final_features]
)[:, 1]

calibration_comparison = pd.DataFrame([
    {
        "Model": "Uncalibrated",
        "Brier": brier_score_loss(
            validation_cancel["NoPurchaseNext180Days"],
            uncalibrated_validation_prob
        ),
        "LogLoss": log_loss(
            validation_cancel["NoPurchaseNext180Days"],
            uncalibrated_validation_prob
        )
    },
    {
        "Model": "Sigmoid calibrated",
        "Brier": brier_score_loss(
            validation_cancel["NoPurchaseNext180Days"],
            calibrated_validation_prob
        ),
        "LogLoss": log_loss(
            validation_cancel["NoPurchaseNext180Days"],
            calibrated_validation_prob
        )
    }
])

calibration_comparison

In [ ]:
prob_true_val, prob_pred_val = calibration_curve(
    validation_cancel["NoPurchaseNext180Days"],
    calibrated_validation_prob,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.plot(
    prob_pred_val,
    prob_true_val,
    marker="o",
    label="Sigmoid calibrated"
)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed proportion")
plt.title("Validation calibration curve")
plt.legend()
plt.show()

## 16A. Freeze the capacity threshold after calibration

The operational probability used downstream is the **calibrated** probability. Therefore the validation-equivalent threshold for the 100-customer capacity is recomputed from the calibrated validation probabilities, not the uncalibrated scores.

This threshold is frozen before the final test is evaluated.

In [ ]:
capacity_threshold = np.sort(
    calibrated_validation_prob
)[-RETENTION_CAPACITY]

capacity_threshold_result = pd.DataFrame([{
    "Capacity": RETENTION_CAPACITY,
    "FrozenValidationThreshold": capacity_threshold,
    "ValidationCustomersSelected": int(
        (calibrated_validation_prob >= capacity_threshold).sum()
    )
}])

capacity_threshold_result

### Calibration decision

Sigmoid calibration is retained if it improves or preserves validation Brier/log loss without harming discrimination. The final test remains untouched until the next section.

# 17. FINAL MODEL FREEZE

Everything above this point is experimentation and decision support.

**The methodology is now frozen:**

- target: `NoPurchaseNext180Days`;
- cohort: >=1 valid purchase in the 180-day observation window;
- train: Jun + Sep 2010;
- validation: Dec 2010;
- March 2011: diagnostic-only;
- final test: Jun 2011;
- features: core + cancellation features;
- redundant gap fields removed;
- `log1p` count/value transformation;
- Logistic Regression;
- sigmoid calibration;
- production queue: capacity-based EVaR ranking;
- final test is not used for any selection or tuning.

# 18. Untouched final temporal test

The frozen model is evaluated on the June 2011 snapshot exactly once for final performance reporting.

No feature selection, threshold selection, calibration choice, or model choice is changed after observing these results.

In [ ]:
final_test_prob = calibrated_candidate.predict_proba(
    test_cancel[final_features]
)[:, 1]

final_test_metrics = evaluate_probabilities(
    test_cancel["NoPurchaseNext180Days"],
    final_test_prob,
    threshold=capacity_threshold
)

final_test_probability_metrics = pd.DataFrame([
    {
        "Metric": "Brier",
        "Value": brier_score_loss(
            test_cancel["NoPurchaseNext180Days"],
            final_test_prob
        )
    },
    {
        "Metric": "LogLoss",
        "Value": log_loss(
            test_cancel["NoPurchaseNext180Days"],
            final_test_prob
        )
    },
    {
        "Metric": "ROC-AUC",
        "Value": roc_auc_score(
            test_cancel["NoPurchaseNext180Days"],
            final_test_prob
        )
    },
    {
        "Metric": "PR-AUC",
        "Value": average_precision_score(
            test_cancel["NoPurchaseNext180Days"],
            final_test_prob
        )
    }
])

print("Final test classification metrics:")
print(pd.Series(final_test_metrics))

print("\nFinal test probability metrics:")
display(final_test_probability_metrics)

## 19. History-group final-test performance

The aggregate metric is accompanied by separate results for single-purchase and established customers so that the model does not hide materially different behavior behind one number.

In [ ]:
history_test_metrics = []

for group_name, group_value in [
    ("Single-purchase", 1),
    ("Established", 0)
]:
    mask = test_cancel["SinglePurchaseHistory"].eq(group_value)
    y_group = test_cancel.loc[mask, "NoPurchaseNext180Days"]
    p_group = final_test_prob[mask]

    history_test_metrics.append({
        "HistoryGroup": group_name,
        "Customers": int(mask.sum()),
        "NoReturnRate": float(y_group.mean()),
        "ROC-AUC": roc_auc_score(y_group, p_group),
        "PR-AUC": average_precision_score(y_group, p_group),
        "Brier": brier_score_loss(y_group, p_group)
    })

history_test_metrics = pd.DataFrame(history_test_metrics)
history_test_metrics

In [ ]:
prob_true_test, prob_pred_test = calibration_curve(
    test_cancel["NoPurchaseNext180Days"],
    final_test_prob,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.plot(
    prob_pred_test,
    prob_true_test,
    marker="o",
    label="Frozen calibrated model"
)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed proportion")
plt.title("Final temporal test calibration curve")
plt.legend()
plt.show()

# 20. Final-test uncertainty

Bootstrap intervals are calculated on the untouched temporal test set. The test labels are not used to change the model.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
y_test_array = test_cancel["NoPurchaseNext180Days"].to_numpy()
p_test_array = final_test_prob

bootstrap_auc = []
bootstrap_precision = []

for _ in range(1000):
    idx = rng.integers(0, len(y_test_array), len(y_test_array))
    y_b = y_test_array[idx]
    p_b = p_test_array[idx]

    if len(np.unique(y_b)) < 2:
        continue

    bootstrap_auc.append(roc_auc_score(y_b, p_b))

    pred_b = (p_b >= capacity_threshold).astype(int)
    bootstrap_precision.append(
        precision_score(y_b, pred_b, zero_division=0)
    )

bootstrap_uncertainty = pd.DataFrame([
    {
        "Metric": "ROC-AUC",
        "PointEstimate": roc_auc_score(y_test_array, p_test_array),
        "Lower95": np.percentile(bootstrap_auc, 2.5),
        "Upper95": np.percentile(bootstrap_auc, 97.5)
    },
    {
        "Metric": "Precision at validation-capacity threshold",
        "PointEstimate": precision_score(
            y_test_array,
            p_test_array >= capacity_threshold,
            zero_division=0
        ),
        "Lower95": np.percentile(bootstrap_precision, 2.5),
        "Upper95": np.percentile(bootstrap_precision, 97.5)
    }
])

bootstrap_uncertainty

# 21. Production refit protocol

Only now — after final test evaluation — do we create the production model.

**Production fitting data = Train + Validation.**

The final June 2011 test set is never used for production fitting.

The production model is therefore allowed to learn from all approved development data while preserving the final test as a historical evaluation record.

In [ ]:
development_data = pd.concat(
    [train_cancel, validation_cancel],
    ignore_index=True
)

production_model = CalibratedClassifierCV(
    Pipeline([
        (
            "feature_transform",
            ColumnTransformer(
                [
                    (
                        "log1p",
                        FunctionTransformer(np.log1p, validate=False),
                        final_log_columns
                    )
                ],
                remainder="passthrough",
                verbose_feature_names_out=False
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]),
    method="sigmoid",
    cv=5
)

production_model.fit(
    development_data[final_features],
    development_data["NoPurchaseNext180Days"]
)

print("Production model refit complete.")
print("Training rows used:", len(development_data))
print("Final test rows used for fitting: 0")

# 22. Production scoring population

The production cutoff is the latest eligible cohort (9 June 2011).

Every eligible customer receives a probability. No `ActualChurn` or future target column is required for production scoring.

This is separate from the final-test evaluation table.

In [ ]:
production_population = test_cancel.copy()

production_population["CalibratedNoReturnProbability"] = (
    production_model.predict_proba(
        production_population[final_features]
    )[:, 1]
)

production_scores = production_population[
    [
        "Customer ID",
        "SnapshotCutoff",
        *final_features,
        "CalibratedNoReturnProbability"
    ]
].copy()

print("Eligible production customers:", len(production_scores))
print("Scored production customers:", len(production_scores))
print("Missing production probabilities:", production_scores["CalibratedNoReturnProbability"].isna().sum())

# 23. Customer Lifetime Value methodology

## Transaction grain decision

The CLV transaction unit is explicitly:

> **one customer-date transaction**

All valid purchase line values on the same customer and calendar date are summed.

This is consistent with `freq='D'` because Lifetimes then interprets each customer-date as one transaction period.

The same customer-date definition is used for:

- frequency,
- recency,
- T,
- monetary value,
- Gamma-Gamma,
- 180-day holdout count,
- 180-day holdout value.

This removes the earlier invoice-vs-date mismatch.

In [ ]:
# Lifetimes is an explicit project dependency. It is NOT installed from the notebook.
from lifetimes.utils import summary_data_from_transaction_data
from lifetimes import BetaGeoFitter, GammaGammaFitter

clv_source = customer_transactions[
    (customer_transactions["InvoiceDate"] <= FINAL_TEST_CUTOFF) &
    (customer_transactions["Transaction_Type"] == "Valid Purchase")
].copy()

clv_source["TransactionValue"] = (
    clv_source["Quantity"] * clv_source["Price"]
)

clv_transactions = (
    clv_source
    .assign(PurchaseDate=clv_source["InvoiceDate"].dt.floor("D"))
    .groupby(["Customer ID", "PurchaseDate"], as_index=False)
    .agg(TransactionValue=("TransactionValue", "sum"))
)

clv_transactions.head()

In [ ]:
clv_summary = summary_data_from_transaction_data(
    clv_transactions,
    customer_id_col="Customer ID",
    datetime_col="PurchaseDate",
    monetary_value_col="TransactionValue",
    observation_period_end=FINAL_TEST_CUTOFF
).reset_index()

clv_summary.head()

## 24. BG/NBD and Gamma-Gamma fitting

The monetary model receives `monetary_value` directly from the same customer-date transaction table.

The model horizon is **12 months**, because Lifetimes' `time` parameter is measured in months when `freq='D'`.

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.1)

bgf.fit(
    clv_summary["frequency"],
    clv_summary["recency"],
    clv_summary["T"]
)

repeat_clv = clv_summary[
    clv_summary["frequency"] > 0
].copy()

gg = GammaGammaFitter(penalizer_coef=0.001)

gg.fit(
    repeat_clv["frequency"],
    repeat_clv["monetary_value"]
)

print("BG/NBD parameters:")
display(bgf.summary)

print("Gamma-Gamma parameters:")
display(gg.summary)

print(
    "Frequency / monetary correlation:",
    repeat_clv[["frequency", "monetary_value"]].corr().iloc[0, 1]
)

## 25. BG/NBD stability experiment

A successful optimizer return is not sufficient evidence of a stable model. Reasonable penalizer values are compared. We retain the defensible configuration and document any instability rather than tuning until the output looks convenient.

In [ ]:
bgnbd_stability_rows = []

for penalizer in [0.01, 0.05, 0.1, 0.2]:
    try:
        candidate = BetaGeoFitter(penalizer_coef=penalizer)
        candidate.fit(
            clv_summary["frequency"],
            clv_summary["recency"],
            clv_summary["T"]
        )
        params = candidate.params_.to_dict()
        bgnbd_stability_rows.append({
            "Penalizer": penalizer,
            "Status": "Success",
            **params
        })
    except Exception as exc:
        bgnbd_stability_rows.append({
            "Penalizer": penalizer,
            "Status": f"Failed: {type(exc).__name__}",
        })

bgnbd_stability = pd.DataFrame(bgnbd_stability_rows)
bgnbd_stability

### Gamma-Gamma assumption diagnostic

The frequency/monetary correlation is reported before accepting the monetary model. A held-out value comparison is performed next.

# 26. CLV holdout validation

The 180-day holdout after the 9 June 2011 cutoff uses the **same customer-date transaction grain**.

Predicted transaction count therefore means predicted purchase days, not invoice-line count.

In [ ]:
holdout_end = FINAL_TEST_CUTOFF + pd.Timedelta(days=OUTCOME_DAYS)

holdout_source = customer_transactions[
    (customer_transactions["InvoiceDate"] > FINAL_TEST_CUTOFF) &
    (customer_transactions["InvoiceDate"] <= holdout_end) &
    (customer_transactions["Transaction_Type"] == "Valid Purchase")
].copy()

holdout_source["TransactionValue"] = (
    holdout_source["Quantity"] * holdout_source["Price"]
)

holdout_transactions = (
    holdout_source
    .assign(PurchaseDate=holdout_source["InvoiceDate"].dt.floor("D"))
    .groupby(["Customer ID", "PurchaseDate"], as_index=False)
    .agg(TransactionValue=("TransactionValue", "sum"))
)

actual_holdout = (
    holdout_transactions
    .groupby("Customer ID")
    .agg(
        ActualPurchaseDateCount=("PurchaseDate", "nunique"),
        ActualPurchaseValue=("TransactionValue", "sum")
    )
    .reset_index()
)

backtest = repeat_clv[
    ["Customer ID", "frequency", "recency", "T", "monetary_value"]
].copy()

backtest["PredictedPurchaseDateCount"] = bgf.predict(
    OUTCOME_DAYS,
    backtest["frequency"],
    backtest["recency"],
    backtest["T"]
)

backtest["PredictedAvgTransactionValue"] = (
    gg.conditional_expected_average_profit(
        backtest["frequency"],
        backtest["monetary_value"]
    )
)

backtest["PredictedPurchaseValue"] = (
    backtest["PredictedPurchaseDateCount"] *
    backtest["PredictedAvgTransactionValue"]
)

backtest_results = backtest.merge(
    actual_holdout,
    on="Customer ID",
    how="left"
).fillna({
    "ActualPurchaseDateCount": 0,
    "ActualPurchaseValue": 0
})

backtest_summary = pd.DataFrame([
    {
        "Metric": "Purchase-date count",
        "PredictedMean": backtest_results["PredictedPurchaseDateCount"].mean(),
        "ActualMean": backtest_results["ActualPurchaseDateCount"].mean(),
        "MAE": mean_absolute_error(
            backtest_results["ActualPurchaseDateCount"],
            backtest_results["PredictedPurchaseDateCount"]
        ),
        "RMSE": mean_squared_error(
            backtest_results["ActualPurchaseDateCount"],
            backtest_results["PredictedPurchaseDateCount"]
        ) ** 0.5
    },
    {
        "Metric": "Purchase value",
        "PredictedMean": backtest_results["PredictedPurchaseValue"].mean(),
        "ActualMean": backtest_results["ActualPurchaseValue"].mean(),
        "MAE": mean_absolute_error(
            backtest_results["ActualPurchaseValue"],
            backtest_results["PredictedPurchaseValue"]
        ),
        "RMSE": mean_squared_error(
            backtest_results["ActualPurchaseValue"],
            backtest_results["PredictedPurchaseValue"]
        ) ** 0.5
    }
])

backtest_summary

### Value-ranking validation

For a prioritization product, ranking quality matters in addition to absolute error. The next table measures how much realized holdout value is captured by the highest predicted-value customers.

In [ ]:
rank_validation = backtest_results[
    ["Customer ID", "PredictedPurchaseValue", "ActualPurchaseValue"]
].copy()

rank_validation["ValueDecile"] = pd.qcut(
    rank_validation["PredictedPurchaseValue"].rank(method="first"),
    10,
    labels=False
) + 1

decile_capture = (
    rank_validation
    .groupby("ValueDecile")
    .agg(
        Customers=("Customer ID", "count"),
        RealizedValue=("ActualPurchaseValue", "sum")
    )
    .sort_values("ValueDecile", ascending=False)
)

decile_capture["ShareOfRealizedValue"] = (
    decile_capture["RealizedValue"] /
    decile_capture["RealizedValue"].sum()
)

decile_capture

# 27. Discount-rate sensitivity

The primary prototype assumption is a 1% monthly discount rate. Because this is a business assumption rather than a dataset fact, rankings are checked at 0%, 0.5%, and 1%.

In [ ]:
discount_rates = [0.00, 0.005, 0.01]
discount_rows = []

for rate in discount_rates:
    predicted = gg.customer_lifetime_value(
        transaction_prediction_model=bgf,
        frequency=repeat_clv["frequency"],
        recency=repeat_clv["recency"],
        T=repeat_clv["T"],
        monetary_value=repeat_clv["monetary_value"],
        time=12,
        discount_rate=rate,
        freq="D"
    )

    discount_rows.append({
        "MonthlyDiscountRate": rate,
        "AverageCLV": predicted.mean(),
        "MedianCLV": predicted.median(),
        "TotalCLV": predicted.sum()
    })

discount_sensitivity = pd.DataFrame(discount_rows)
discount_sensitivity

# 28. FINAL CLV decision

The final CLV output is called `PredictedGrossPurchaseCLV`.

It is **not profit** and not net economic CLV because the model does not incorporate margin, retention cost, refunds, or cancellation costs.

One-time customers remain in the decision layer with:

`CLVStatus = Not Eligible - One-Time Customer`

and use historical valid-purchase value as a fallback proxy.

In [ ]:
repeat_clv["PredictedGrossPurchaseCLV"] = gg.customer_lifetime_value(
    transaction_prediction_model=bgf,
    frequency=repeat_clv["frequency"],
    recency=repeat_clv["recency"],
    T=repeat_clv["T"],
    monetary_value=repeat_clv["monetary_value"],
    time=12,
    discount_rate=0.01,
    freq="D"
)

clv_all = clv_summary.copy()

clv_all["CLVStatus"] = np.where(
    clv_all["frequency"] > 0,
    "Predicted CLV Eligible",
    "Not Eligible - One-Time Customer"
)

historical_purchase_value = (
    clv_transactions.groupby("Customer ID")["TransactionValue"]
    .sum()
    .rename("HistoricalPurchaseValue")
    .reset_index()
)

clv_all = clv_all.merge(
    historical_purchase_value,
    on="Customer ID",
    how="left"
)

clv_all = clv_all.merge(
    repeat_clv[
        ["Customer ID", "PredictedGrossPurchaseCLV"]
    ],
    on="Customer ID",
    how="left"
)

clv_all["ValueForRetention"] = np.where(
    clv_all["CLVStatus"] == "Predicted CLV Eligible",
    clv_all["PredictedGrossPurchaseCLV"],
    clv_all["HistoricalPurchaseValue"]
)

clv_all[[
    "Customer ID",
    "frequency",
    "CLVStatus",
    "HistoricalPurchaseValue",
    "PredictedGrossPurchaseCLV",
    "ValueForRetention"
]].head()

# 29. Retention decision layer

Expected Value at Risk (EVaR) is defined as:

**calibrated probability of no return × value for retention**

Because probability covers 180 days while CLV is a 12-month gross-purchase value, EVaR is explicitly treated as a **prioritization score**, not a direct 180-day revenue forecast.

The queue is capacity-based rather than qcut-based.

In [ ]:
# Use production probabilities generated after final-test evaluation.
retention_base = production_scores[
    ["Customer ID", "SnapshotCutoff", "CalibratedNoReturnProbability"]
].merge(
    clv_all[
        [
            "Customer ID",
            "CLVStatus",
            "HistoricalPurchaseValue",
            "PredictedGrossPurchaseCLV",
            "ValueForRetention"
        ]
    ],
    on="Customer ID",
    how="left"
)

retention_base["ExpectedValueAtRisk"] = (
    retention_base["CalibratedNoReturnProbability"] *
    retention_base["ValueForRetention"]
)

retention_priority_list = (
    retention_base
    .sort_values(
        "ExpectedValueAtRisk",
        ascending=False
    )
    .head(RETENTION_CAPACITY)
    .copy()
)

retention_priority_list["RetentionPriority"] = "Priority"
retention_base["RetentionPriority"] = "Standard"
retention_base.loc[
    retention_base["Customer ID"].isin(
        retention_priority_list["Customer ID"]
    ),
    "RetentionPriority"
] = "Priority"

print("Eligible:", len(retention_base))
print("Priority:", len(retention_priority_list))
print("Customers missing a value proxy:", retention_base["ValueForRetention"].isna().sum())

# 30. Retention experiment / KPI framework

The model does not claim that a ranking automatically creates a successful retention strategy.

Each operational group therefore needs an intervention hypothesis and measurable outcome.

In [ ]:
retention_experiment_table = pd.DataFrame([
    {
        "PriorityRule": "Top EVaR / Priority",
        "Action": "Personalized retention offer or high-touch outreach",
        "ChannelAssumption": "Email + targeted direct outreach",
        "MaxContactCost": "Set from campaign economics before launch",
        "SuccessMetric": "Incremental 30/60/90-day repeat-purchase rate and incremental revenue",
        "EvaluationPeriod": "30/60/90 days"
    },
    {
        "PriorityRule": "High risk but lower value",
        "Action": "Automated re-engagement",
        "ChannelAssumption": "Email / low-cost automation",
        "MaxContactCost": "Low fixed campaign cost",
        "SuccessMetric": "Repeat purchase rate and cost per retained customer",
        "EvaluationPeriod": "30/60 days"
    },
    {
        "PriorityRule": "Standard",
        "Action": "Regular engagement / no high-cost intervention",
        "ChannelAssumption": "Existing lifecycle communication",
        "MaxContactCost": "Existing communication budget",
        "SuccessMetric": "Baseline repeat-purchase rate",
        "EvaluationPeriod": "60/90 days"
    }
])

retention_experiment_table

In [ ]:
retention_kpis = pd.DataFrame([
    {
        "KPI": "Priority customers",
        "Value": len(retention_priority_list)
    },
    {
        "KPI": "Total EVaR captured by priority queue",
        "Value": retention_priority_list["ExpectedValueAtRisk"].sum()
    },
    {
        "KPI": "Share of total EVaR captured by priority queue",
        "Value": (
            retention_priority_list["ExpectedValueAtRisk"].sum() /
            retention_base["ExpectedValueAtRisk"].sum()
        )
    },
    {
        "KPI": "Average EVaR per priority customer",
        "Value": retention_priority_list["ExpectedValueAtRisk"].mean()
    }
])

retention_kpis

# 31. Final customer intelligence artifact

This is the production-facing schema. It intentionally contains no future truth such as `ActualChurn`.

The evaluation datasets remain separate from this production artifact.

In [ ]:
final_customer_intelligence = retention_base[
    [
        "Customer ID",
        "SnapshotCutoff",
        "CLVStatus",
        "CalibratedNoReturnProbability",
        "ValueForRetention",
        "ExpectedValueAtRisk",
        "RetentionPriority"
    ]
].sort_values(
    "ExpectedValueAtRisk",
    ascending=False
).reset_index(drop=True)

final_customer_intelligence.head(20)

In [ ]:
# Final integrity checks
assert final_customer_intelligence["Customer ID"].is_unique
assert final_customer_intelligence["CalibratedNoReturnProbability"].notna().all()
assert final_customer_intelligence["ValueForRetention"].notna().all()
assert "NoPurchaseNext180Days" not in final_customer_intelligence.columns
assert "ActualChurn" not in final_customer_intelligence.columns

print("Final production rows:", len(final_customer_intelligence))
print("Priority rows:", (final_customer_intelligence["RetentionPriority"] == "Priority").sum())
print("Production schema integrity: PASS")

# 32. Final methodological summary

### What the evidence showed

1. Exact duplicates materially affect monetary totals, so the deduplication assumption is documented.
2. Anonymous transactions are separated from identified-customer analysis.
3. Transaction labels are evidence-based.
4. The target is explicitly a 180-day non-return proxy.
5. The operational cohort contains customers active in the observation window.
6. Multiple historical snapshots provide temporal evaluation.
7. Redundant purchase-gap features were removed after ablation.
8. Skewed count/value features were tested and the selected transformation is documented.
9. Recency-only performance is reported as the simple business baseline.
10. Cancellation behavior was tested and retained only after validation/stability evidence.
11. Calibration was evaluated before the model was frozen.
12. Threshold behavior was evaluated on validation; production uses capacity-based EVaR ranking.
13. Final test evaluation occurs once after freeze.
14. Final production refitting uses Train + Validation only.
15. CLV uses one customer-date transaction definition consistently.
16. CLV uses a 12-month horizon and is validated on a 180-day holdout.
17. One-time customers remain in the decision layer with an explicit fallback.
18. EVaR is presented as a prioritization score, not a direct revenue forecast.

### Final separation

**Research/evaluation:** temporal validation + final test metrics  
**Production:** refit on approved development data + full eligible population scoring  
**Decision layer:** calibrated non-return probability × gross-purchase value  
**Deployment:** offline artifacts + Streamlit app; the app does not retrain models.

# 33. Reproducibility outputs

The following objects are intended for export by the deployment workflow:

- `final_customer_intelligence.csv`
- `final_customer_intelligence.parquet`
- `model_metadata.json`
- `churn_pipeline.joblib`
- `clv_summary.parquet`
- `retention_experiment_table.csv`
- `retention_kpis.csv`

The notebook is the analytical record; deployment should consume saved artifacts rather than retraining on Streamlit startup.